In [1]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

print(pd.__version__)
print(np.__version__)

3.0.2
2.4.3


In [4]:
import pandas as pd
from rapidfuzz import process, fuzz
from pathlib import Path

file_11k = Path(r"/11kadmins.xlsx")
file_55k = Path(r"/UBOS55k.xlsx")

out_file = Path(r"C:\Users\Carl\Desktop\11kadmins_matched.xlsx")

admins = pd.read_excel(file_11k)
ubos = pd.read_excel(file_55k)

def clean_col(c):
    return (
        str(c)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

admins.columns = [clean_col(c) for c in admins.columns]
ubos.columns = [clean_col(c) for c in ubos.columns]

print("11k columns:", admins.columns.tolist())
print("UBOS columns:", ubos.columns.tolist())


admins = admins.rename(columns={
    "name": "name_11k",
    "district": "district_11k",
    "constituency": "constituency_11k",
    "subcounty": "subcounty_11k"
})

ubos = ubos.rename(columns={
    "village": "village_ubos",
    "district": "district_ubos",
    "constituency": "constituency_ubos",
    "subcounty": "subcounty_ubos",
    "parish": "parish_ubos"
})

print("11k after rename:", admins.columns.tolist())
print("UBOS after rename:", ubos.columns.tolist())


def clean_text(x):
    if pd.isna(x):
        return ""
    return (
        str(x)
        .upper()
        .strip()
        .replace("  ", " ")
    )


for col in ["name_11k", "district_11k", "constituency_11k", "subcounty_11k"]:
    admins[col + "_clean"] = admins[col].apply(clean_text)

for col in ["village_ubos", "district_ubos", "constituency_ubos", "subcounty_ubos"]:
    ubos[col + "_clean"] = ubos[col].apply(clean_text)


def best_village_match(row, filter_cols=None):
    """
    Find best UBOS village match for a row in 11kadmins.
    filter_cols controls which admin levels must match exactly.
    """

    candidates = ubos.copy()

    if filter_cols is not None:
        for left_col, right_col in filter_cols:
            left_value = row[left_col]

            if left_value == "":
                return pd.Series([None, None, None, None, None, None])

            candidates = candidates[candidates[right_col] == left_value]

    if candidates.empty or row["name_11k_clean"] == "":
        return pd.Series([None, None, None, None, None, None])

    choices = candidates["village_ubos_clean"].tolist()

    match = process.extractOne(
        row["name_11k_clean"],
        choices,
        scorer=fuzz.WRatio
    )

    if match is None:
        return pd.Series([None, None, None, None, None, None])

    matched_name_clean = match[0]
    score = match[1]

    matched_row = candidates[candidates["village_ubos_clean"] == matched_name_clean].iloc[0]

    return pd.Series([
        matched_row["village_ubos"],
        score,
        matched_row["district_ubos"],
        matched_row["constituency_ubos"],
        matched_row["subcounty_ubos"],
        matched_row["parish_ubos"]
    ])


admins[
    [
        "namematch",
        "namematch_score",
        "namematch_district",
        "namematch_constituency",
        "namematch_subcounty",
        "namematch_parish"
    ]
] = admins.apply(
    best_village_match,
    axis=1,
    filter_cols=None
)


admins[
    [
        "namematch_districtmatch",
        "namematch_districtmatch_score",
        "districtmatch_district",
        "districtmatch_constituency",
        "districtmatch_subcounty",
        "districtmatch_parish"
    ]
] = admins.apply(
    best_village_match,
    axis=1,
    filter_cols=[
        ("district_11k_clean", "district_ubos_clean")
    ]
)



admins[
    [
        "namematch_district_constituencymatch",
        "namematch_district_constituencymatch_score",
        "district_constituencymatch_district",
        "district_constituencymatch_constituency",
        "district_constituencymatch_subcounty",
        "district_constituencymatch_parish"
    ]
] = admins.apply(
    best_village_match,
    axis=1,
    filter_cols=[
        ("district_11k_clean", "district_ubos_clean"),
        ("constituency_11k_clean", "constituency_ubos_clean")
    ]
)


admins[
    [
        "namematch_district_constituency_subcountymatch",
        "namematch_district_constituency_subcountymatch_score",
        "district_constituency_subcountymatch_district",
        "district_constituency_subcountymatch_constituency",
        "district_constituency_subcountymatch_subcounty",
        "district_constituency_subcountymatch_parish"
    ]
] = admins.apply(
    best_village_match,
    axis=1,
    filter_cols=[
        ("district_11k_clean", "district_ubos_clean"),
        ("constituency_11k_clean", "constituency_ubos_clean"),
        ("subcounty_11k_clean", "subcounty_ubos_clean")
    ]
)


score_cols = [
    "namematch_score",
    "namematch_districtmatch_score",
    "namematch_district_constituencymatch_score",
    "namematch_district_constituency_subcountymatch_score"
]

for col in score_cols:
    admins[col.replace("_score", "_flag")] = admins[col].apply(
        lambda x: "KEEP" if pd.notna(x) and x >= 90
        else "REVIEW" if pd.notna(x) and x >= 80
        else "NO MATCH"
    )


clean_cols = [c for c in admins.columns if c.endswith("_clean")]
admins = admins.drop(columns=clean_cols)

admins.to_excel(out_file, index=False)

print(f"Done. Saved to: {out_file}")

11k columns: ['index', 'name', 'district', 'constituency', 'subcounty']
UBOS columns: ['district', 'constituency', 'subcounty', 'parish', 'village']
11k after rename: ['index', 'name_11k', 'district_11k', 'constituency_11k', 'subcounty_11k']
UBOS after rename: ['district_ubos', 'constituency_ubos', 'subcounty_ubos', 'parish_ubos', 'village_ubos']
Done. Saved to: C:\Users\Carl\Desktop\11kadmins_matched.xlsx
